# 3.2: Diagnosing with activation and gradient statistics

**Gradients.** After `loss.backward()`, record the L2 norm of the gradient for each parameter tensor, or the mean absolute gradient per layer. With vanishing gradients, early-layer gradient norms are orders of magnitude smaller than late-layer norms. With exploding gradients, some norms are huge and may already contain Inf.

A few lines of PyTorch are enough to see the problem. The function below builds a deep MLP, runs one forward and backward pass, and returns the gradient norm of each hidden layer's weight matrix:


In [ ]:
import torch
import torch.nn as nn

def layer_grad_norms(activation, init, depth=20, width=256):
    torch.manual_seed(0)
    layers = []
    for _ in range(depth):
        lin = nn.Linear(width, width)
        init(lin.weight)
        nn.init.zeros_(lin.bias)
        layers += [lin, activation()]
    model = nn.Sequential(*layers, nn.Linear(width, 10))
    x, y = torch.randn(64, width), torch.randint(0, 10, (64,))
    nn.functional.cross_entropy(model(x), y).backward()
    return [m.weight.grad.norm().item() for m in model if isinstance(m, nn.Linear)][:-1]

sig = layer_grad_norms(nn.Sigmoid, lambda w: nn.init.xavier_normal_(w))
relu = layer_grad_norms(nn.ReLU, lambda w: nn.init.kaiming_normal_(w, nonlinearity="relu"))
print("sigmoid+Xavier: first %.2e  last %.2e" % (sig[0], sig[-1]))
print("ReLU+He:        first %.2e  last %.2e" % (relu[0], relu[-1]))
